# Mobile Product A/B Testing & Revenue Analytics

## Complete Experiment Analysis

**Project type:** Independent analytics case study  
**Dataset:** Gamelytics: Mobile Analytics Challenge  
**Primary analytical unit:** User  
**Primary business question:** Does the test treatment improve monetization enough to justify rollout?

### Analytical approach

**Data → Experiment validation → KPI analysis → Statistical testing → Sensitivity analysis → Business decision**

This project uses a public mobile experimentation dataset to demonstrate practical skills in data analysis, SQL-oriented data thinking, statistical inference, and business storytelling.

> **Scope:** This is an independent portfolio case study. Conclusions are based only on the available dataset and are not presented as professional employment experience.


## 1. Business Context

A mobile product team is evaluating two experimental variants. The business needs to understand whether the test variant creates a sufficiently strong improvement in monetization to support a rollout decision.

The key analytical challenge is that **conversion and revenue can move in different directions**. A treatment may produce fewer paying users while generating more revenue from the users who do pay.

Therefore, this analysis evaluates:

- **Conversion rate** — how many users become paying users
- **ARPU** — revenue per experiment user
- **ARPPU** — revenue per paying user
- **Statistical significance** — whether observed differences are unlikely to be sampling noise
- **Business significance** — whether the size and direction of the effect support a practical decision


## 2. Data Source & Analytical Scope

The source archive contains three tables:

| Table | Grain | Main purpose |
|---|---|---|
| `ab_test.csv` | One row per experiment user | Assignment and user-level revenue |
| `reg_data.csv` | One row per registered user | Registration context |
| `auth_data.csv` | Multiple activity records per user | Authentication/activity context |

The **primary experiment analysis is anchored to `ab_test.csv`** because it contains the assignment variable and revenue outcome at the same user-level grain.

The registration and activity tables are used only for supporting population/context checks. They are not forced into the causal experiment analysis when the available structure does not support it.


In [ ]:
# ============================================================
# CELL 1 — LOAD THE RAW DATA
# ============================================================
# WHY:
# The notebook starts from the original ZIP archive so the
# analysis is reproducible from the source data.
#
# The semicolon delimiter is specified explicitly because these
# source CSVs are semicolon-delimited.

from pathlib import Path
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

DATA_PATH = Path(
    r"E:\DA PORTFOLIO\DA-Projects\Product AB Testing & Revenue Analytics\data\raw\archive.zip"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

with zipfile.ZipFile(DATA_PATH, "r") as z:
    required = {"ab_test.csv", "reg_data.csv", "auth_data.csv"}
    missing = required - set(z.namelist())

    if missing:
        raise FileNotFoundError(f"Missing source file(s): {sorted(missing)}")

    ab_test = pd.read_csv(z.open("ab_test.csv"), sep=";")
    reg_data = pd.read_csv(z.open("reg_data.csv"), sep=";")
    auth_data = pd.read_csv(z.open("auth_data.csv"), sep=";")

print("Source loaded successfully.")
print(f"ab_test   : {len(ab_test):,} rows × {ab_test.shape[1]} columns")
print(f"reg_data  : {len(reg_data):,} rows × {reg_data.shape[1]} columns")
print(f"auth_data : {len(auth_data):,} rows × {auth_data.shape[1]} columns")


## 3. Schema & Data Integrity

Before calculating business metrics, we validate the basic structure of the data.

The important question is not whether every table has one row per user. The tables have different grains. The critical requirement for the experiment is specifically:

> **One experiment record per user in `ab_test`.**

We also check missing values and exact duplicate rows because either can distort user-level experiment metrics.


In [ ]:
# ============================================================
# CELL 2 — CORE DATA QUALITY CHECKS
# ============================================================
# WHY:
# We keep the audit focused on checks that can directly affect
# the experiment analysis: missing values, duplicate rows, and
# the grain of the primary experiment table.

quality = []

for name, df in {
    "ab_test": ab_test,
    "reg_data": reg_data,
    "auth_data": auth_data
}.items():
    quality.append({
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "null_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

quality = pd.DataFrame(quality)
display(quality)

ab_users = ab_test["user_id"].nunique()

print(f"ab_test rows: {len(ab_test):,}")
print(f"Unique experiment users: {ab_users:,}")
print(f"One row per experiment user: {len(ab_test) == ab_users}")

if len(ab_test) != ab_users:
    raise ValueError("The experiment table is not one row per user.")


### Data-quality interpretation

The primary experiment table is structurally suitable for user-level analysis when:

- there are no missing experiment fields,
- there are no exact duplicate rows, and
- the number of rows equals the number of unique experiment users.

The activity table is intentionally different: multiple activity records per user are expected and are not treated as duplicates.


## 4. Experiment Population & Allocation

We now inspect the assignment variable and quantify the experiment population.

The allocation check is descriptive. A formal sample-ratio-mismatch test is included later with the statistical analysis so that the experiment-quality decision is made in one place.


In [ ]:
# ============================================================
# CELL 3 — EXPERIMENT GROUP ALLOCATION
# ============================================================
# WHY:
# Before comparing outcomes, we need to verify the actual groups
# and check whether the user allocation is approximately balanced.

group_summary = (
    ab_test.groupby("testgroup")["user_id"]
    .nunique()
    .reset_index(name="users")
)

group_summary["share_pct"] = (
    group_summary["users"] / group_summary["users"].sum() * 100
)

display(group_summary)

print(f"Observed groups: {sorted(ab_test['testgroup'].unique())}")
print("Control group: A")
print("Test group: B")


## 5. Core Business KPIs

These metrics answer the first business question: **what changed between the two groups?**

We use three complementary monetization views:

- **Conversion:** breadth of monetization
- **ARPU:** total monetization per experiment user
- **ARPPU:** monetization intensity among paying users

The calculations below are descriptive. Statistical significance is assessed separately.


In [ ]:
# ============================================================
# CELL 4 — CORE EXPERIMENT KPIs
# ============================================================
# WHY:
# Looking at conversion, ARPU and ARPPU together prevents a
# single KPI from hiding an important trade-off.

kpi = (
    ab_test.assign(payer=ab_test["revenue"] > 0)
    .groupby("testgroup")
    .agg(
        users=("user_id", "nunique"),
        paying_users=("payer", "sum"),
        total_revenue=("revenue", "sum")
    )
    .reset_index()
)

kpi["conversion_rate"] = kpi["paying_users"] / kpi["users"]
kpi["arpu"] = kpi["total_revenue"] / kpi["users"]
kpi["arppu"] = kpi["total_revenue"] / kpi["paying_users"]

display(
    kpi.assign(
        conversion_pct=kpi["conversion_rate"] * 100
    )[
        [
            "testgroup", "users", "paying_users",
            "conversion_pct", "total_revenue", "arpu", "arppu"
        ]
    ].round(2)
)

control = kpi.loc[kpi["testgroup"] == "a"].iloc[0]
test = kpi.loc[kpi["testgroup"] == "b"].iloc[0]

comparison = pd.DataFrame({
    "metric": ["Conversion Rate", "ARPU", "ARPPU", "Total Revenue"],
    "Control_A": [
        control["conversion_rate"], control["arpu"],
        control["arppu"], control["total_revenue"]
    ],
    "Test_B": [
        test["conversion_rate"], test["arpu"],
        test["arppu"], test["total_revenue"]
    ]
})

comparison["absolute_difference"] = (
    comparison["Test_B"] - comparison["Control_A"]
)
comparison["relative_lift_pct"] = (
    comparison["absolute_difference"] / comparison["Control_A"] * 100
)

display(comparison.round(4))


### Descriptive finding

The observed results create a meaningful trade-off:

- Test B has **lower conversion** than Control A.
- Test B has **higher ARPU**.
- Test B also has **higher ARPPU**.

This is exactly why the experiment cannot be judged from one KPI. The next sections determine whether these observed differences are statistically credible.


## 6. Revenue Distribution Diagnostic

Revenue is not normally distributed: most users generate zero revenue while a small number of users contribute substantial amounts.

This matters because the mean is sensitive to high-value observations. We therefore inspect the distribution before choosing and interpreting revenue inference.


In [ ]:
# ============================================================
# CELL 5 — REVENUE DISTRIBUTION DIAGNOSTIC
# ============================================================
# WHY:
# We need to understand the shape of revenue before interpreting
# a mean comparison. The percentile table and one visual provide
# enough evidence without creating unnecessary charts.

revenue_stats = ab_test["revenue"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
)

display(revenue_stats.to_frame("revenue"))

paying_share = (ab_test["revenue"] > 0).mean()
print(f"Paying-user share: {paying_share:.3%}")

plt.figure(figsize=(9, 4.5))
plt.hist(np.log1p(ab_test["revenue"]), bins=70)
plt.title("User Revenue Distribution — log(1 + revenue)")
plt.xlabel("log(1 + revenue)")
plt.ylabel("Users")
plt.tight_layout()
plt.show()


### Distribution interpretation

The revenue distribution is strongly zero-inflated and right-skewed. Therefore:

- conversion should be analysed as a binary outcome;
- revenue should be analysed with a method that does not require equal variances;
- a sensitivity analysis is useful because a small number of high-value users can influence mean revenue.

We will not treat a normality assumption as automatically satisfied.


## 7. Supporting Population Linkage

The registration and authentication tables provide additional user/activity context.

We do not use them to redefine the primary experiment population. Instead, we quantify how much of the experiment population can be linked to those supporting tables.

This is important because silently dropping unmatched users could change the experiment population and introduce selection into the analysis.


In [ ]:
# ============================================================
# CELL 6 — SUPPORTING TABLE LINKAGE
# ============================================================
# WHY:
# We quantify linkage coverage without changing the primary
# experiment population.

registered_ids = set(reg_data["uid"])
active_ids = set(auth_data["uid"])

linkage = pd.DataFrame({
    "population": [
        "Experiment users",
        "Experiment users found in registration data",
        "Experiment users found in activity data"
    ],
    "users": [
        ab_test["user_id"].nunique(),
        ab_test["user_id"].isin(registered_ids).sum(),
        ab_test["user_id"].isin(active_ids).sum()
    ]
})

linkage["coverage_pct"] = (
    linkage["users"] / len(ab_test) * 100
)

display(linkage.round(2))


## 8. Statistical Experiment Analysis

### 8.1 Hypothesis framework

We evaluate two separate outcomes.

**Conversion**

- H₀: conversion rate is equal between A and B
- H₁: conversion rate differs between A and B

**ARPU**

- H₀: mean revenue per experiment user is equal between A and B
- H₁: mean revenue per experiment user differs between A and B

We use a two-sided significance level of **α = 0.05**.

The goal is not to find a significant result at any cost. The goal is to determine whether the evidence is strong enough to support a business decision.


In [ ]:
# ============================================================
# CELL 7 — SAMPLE RATIO MISMATCH + CONVERSION TEST
# ============================================================
# WHY:
# First, we test whether the observed A/B allocation differs
# materially from the expected 50/50 split.
#
# Then we test conversion because it is a binary user outcome.
# A two-proportion z-test is appropriate for comparing the two
# independent group proportions.

from scipy.stats import chisquare, norm

alpha = 0.05

observed_users = np.array([len(a) if "a" in ab_test["testgroup"].values else 0,
                           len(b) if "b" in ab_test["testgroup"].values else 0])

srm = chisquare(observed_users)

print("Sample Ratio Mismatch Test")
print(f"Chi-square statistic: {srm.statistic:.4f}")
print(f"p-value: {srm.pvalue:.4f}")
print(f"Conclusion: {'No significant allocation mismatch' if srm.pvalue >= alpha else 'Potential allocation mismatch'}")


# Conversion test
a_payers = int((a := ab_test[ab_test["testgroup"] == "a"])["revenue"].gt(0).sum())
b_payers = int((b := ab_test[ab_test["testgroup"] == "b"])["revenue"].gt(0).sum())

a_n, b_n = len(a), len(b)
p_a, p_b = a_payers / a_n, b_payers / b_n

pooled_p = (a_payers + b_payers) / (a_n + b_n)
se_null = np.sqrt(pooled_p * (1 - pooled_p) * (1/a_n + 1/b_n))

z = (p_b - p_a) / se_null
p_value = 2 * norm.sf(abs(z))

# Wald confidence interval for the observed B - A difference.
diff = p_b - p_a
se_diff = np.sqrt(
    p_a * (1 - p_a) / a_n +
    p_b * (1 - p_b) / b_n
)
ci_low = diff - 1.96 * se_diff
ci_high = diff + 1.96 * se_diff

print("\nConversion Test")
print(f"Control conversion: {p_a:.3%}")
print(f"Test conversion:    {p_b:.3%}")
print(f"Absolute difference (B - A): {diff:.3%}")
print(f"Relative lift: {diff / p_a:.2%}")
print(f"z-statistic: {z:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"95% CI for B - A: [{ci_low:.3%}, {ci_high:.3%}]")
print(f"Statistically significant at 5%: {p_value < alpha}")


### Conversion interpretation

The formal conversion test shows a statistically significant decline in Test B.

- Control conversion: **0.954%**
- Test conversion: **0.891%**
- Relative change: **−6.64%**
- p-value: **0.0350**
- 95% CI for B − A: **−0.122% to −0.004%**

Because the confidence interval remains below zero, the observed conversion decline is statistically significant at the 5% level.

This is a negative experiment signal, not evidence of a successful treatment.

In [ ]:
# ============================================================
# CELL 8 — ARPU TEST + SENSITIVITY ANALYSIS
# ============================================================
# WHY:
# Revenue is highly skewed and the two groups do not have equal
# variance. Welch's t-test is therefore preferable to the
# equal-variance t-test for the primary mean-revenue comparison.
#
# We also run a Mann–Whitney U test as a sensitivity analysis.
# The second test does not rely on the same mean/variance structure,
# so agreement or disagreement helps us judge robustness.

a_rev = a["revenue"].to_numpy()
b_rev = b["revenue"].to_numpy()

welch = stats.ttest_ind(a_rev, b_rev, equal_var=False)

# Welch confidence interval for mean difference B - A.
mean_diff = b_rev.mean() - a_rev.mean()

var_a = a_rev.var(ddof=1)
var_b = b_rev.var(ddof=1)

se_welch = np.sqrt(var_a / len(a_rev) + var_b / len(b_rev))

df_welch = (
    (var_a / len(a_rev) + var_b / len(b_rev)) ** 2
    /
    (
        (var_a / len(a_rev)) ** 2 / (len(a_rev) - 1)
        +
        (var_b / len(b_rev)) ** 2 / (len(b_rev) - 1)
    )
)

t_critical = stats.t.ppf(1 - alpha/2, df_welch)

arpu_ci_low = mean_diff - t_critical * se_welch
arpu_ci_high = mean_diff + t_critical * se_welch

# Sensitivity check using a non-parametric comparison.
mann_whitney = stats.mannwhitneyu(
    a_rev,
    b_rev,
    alternative="two-sided"
)

print("ARPU / Revenue Analysis")
print(f"Control ARPU: {a_rev.mean():.4f}")
print(f"Test ARPU:    {b_rev.mean():.4f}")
print(f"Difference B - A: {mean_diff:.4f}")
print(f"Relative lift: {mean_diff / a_rev.mean():.2%}")
print(f"Welch t-statistic: {welch.statistic:.4f}")
print(f"Welch p-value: {welch.pvalue:.4f}")
print(f"95% CI for B - A: [{arpu_ci_low:.4f}, {arpu_ci_high:.4f}]")
print(f"Mann–Whitney p-value: {mann_whitney.pvalue:.4f}")


## 9. Statistical vs. Business Significance

The experiment should not be judged by statistical significance alone.

A useful decision requires both:

1. **Statistical evidence** — is the observed effect sufficiently distinguishable from sampling variation?
2. **Business meaning** — is the direction and magnitude attractive enough to justify a rollout?

For this dataset, the evidence is mixed:

- Test B generates **5.26% higher observed ARPU** and **12.75% higher observed ARPPU**.
- Test B has **6.64% lower conversion** than Control A.
- The conversion decline is statistically significant (**p = 0.0350**).
- The ARPU increase is not statistically significant (**Welch p ≈ 0.533**).
- The non-parametric sensitivity check also does not reach the 5% threshold (**Mann–Whitney p ≈ 0.063**).
- Revenue is highly zero-inflated and right-skewed, so the observed ARPU increase should be interpreted cautiously.

Taken together, the evidence does not provide strong support for an immediate rollout.

In [ ]:
# ============================================================
# CELL 9 — EXECUTIVE DECISION
# ============================================================
# WHY:
# The final decision should be rule-based rather than driven by
# whichever KPI looks most attractive.
#
# We require evidence of a meaningful improvement before rollout.
# If the evidence is mixed or inconclusive, the appropriate action
# is to iterate or retest rather than force a positive conclusion.

conversion_worse_and_significant = (
    p_value < alpha and p_b < p_a
)

revenue_not_significant = welch.pvalue >= alpha

if conversion_worse_and_significant and revenue_not_significant:
    decision = "RETEST / ITERATE"
    rationale = (
        "Test B reduces conversion with statistically significant evidence, "
        "while the observed ARPU increase is not statistically significant."
    )
elif welch.pvalue < alpha and mean_diff > 0 and p_b >= p_a:
    decision = "ROLL OUT"
    rationale = (
        "The test shows statistically significant revenue improvement "
        "without a statistically significant conversion deterioration."
    )
elif mean_diff < 0 and welch.pvalue < alpha:
    decision = "DO NOT ROLL OUT"
    rationale = (
        "The test produces statistically significant revenue deterioration."
    )
else:
    decision = "RETEST / ITERATE"
    rationale = (
        "The evidence is not sufficiently strong and consistent to support "
        "a confident rollout decision."
    )

print("=" * 70)
print("FINAL EXPERIMENT DECISION")
print("=" * 70)
print(f"\nDecision: {decision}")
print(f"\nRationale:\n{rationale}")


## 10. Supporting Activity Context

The registration and authentication tables contain timestamp information and can support additional product-behaviour analysis.

For this core experiment case study, we deliberately do **not** manufacture retention, churn, LTV, CAC, or ROAS metrics from fields that do not directly support those definitions.

This keeps the experiment conclusion tied to the actual evidence available in the dataset.


In [ ]:
# ============================================================
# CELL 10 — TIMESTAMP COVERAGE CHECK
# ============================================================
# WHY:
# We only need the timestamp range here to document the supporting
# data coverage. A full retention model is outside the core
# experiment question and should not be added without a defensible
# experiment-level definition.

reg_dates = pd.to_datetime(reg_data["reg_ts"], unit="s")
auth_dates = pd.to_datetime(auth_data["auth_ts"], unit="s")

timestamp_summary = pd.DataFrame({
    "table": ["reg_data", "auth_data"],
    "start": [reg_dates.min(), auth_dates.min()],
    "end": [reg_dates.max(), auth_dates.max()]
})

display(timestamp_summary)


## 11. Key Findings

### Experiment quality

- The experiment table contains **404,770 unique users**.
- There are **no missing cells** across the three source tables.
- There are **no exact duplicate rows**.
- A/B allocation is approximately balanced: **49.93% vs 50.07%**.
- The formal sample-ratio-mismatch test does not indicate a significant allocation problem (**p = 0.3754**).

### Monetization

- Control A: **0.954% conversion**, **25.41 ARPU**, **2,664.00 ARPPU**
- Test B: **0.891% conversion**, **26.75 ARPU**, **3,003.66 ARPPU**
- Test B therefore has lower conversion but higher observed ARPU and ARPPU.

### Statistical evidence

- Conversion declines by **6.64% relative to Control A**, with **p = 0.0350** and a 95% CI for B − A of **−0.122% to −0.004%**.
- ARPU is **5.26% higher descriptively**, but the Welch test is not statistically significant (**p ≈ 0.533**).
- The Mann–Whitney sensitivity check is also above the 5% threshold (**p ≈ 0.063**).
- Revenue is strongly zero-inflated and right-skewed, so the observed mean uplift should not be treated as definitive evidence of a treatment effect.

### Business decision

**RETEST / ITERATE**

The evidence does not justify a confident rollout because Test B produces a statistically significant conversion decline while its observed ARPU improvement is not statistically significant.

## 12. Limitations

1. The analysis is based on a public dataset rather than production company data.
2. The experiment table does not expose a detailed offer-design field or explicit exposure timestamp.
3. Revenue observation-window details are not directly encoded in the experiment table.
4. A portion of experiment users cannot be linked to the registration/activity population; the primary experiment population is therefore kept anchored to `ab_test.csv`.
5. Revenue is highly skewed and concentrated, so mean-based revenue comparisons require sensitivity analysis.
6. Subscription-specific metrics such as LTV, CAC, churn, and ROAS are not claimed unless the underlying data supports a defensible definition.


## 13. Executive Takeaway

> **Test B does not provide sufficient evidence for immediate rollout.**

Test B produced a **5.26% higher observed ARPU** and **12.75% higher observed ARPPU**, but conversion fell by approximately **6.64% relative to Control A**, and that conversion decline is statistically significant.

The ARPU increase is not statistically significant, and the sensitivity check reaches the same practical conclusion at the 5% threshold.

The appropriate business response is therefore:

### **RETEST / ITERATE**

A follow-up experiment should investigate whether the potential monetization benefit can be retained without reducing the number of users who convert.

This conclusion is based on the observed experiment data and statistical evidence rather than on a forced positive result.

## 14. Project Handoff

This notebook is the core analytical engine for the case study.

### Next project deliverables

**SQL**
- Reproduce experiment KPIs
- Demonstrate joins across the available tables
- Use CTEs and conditional aggregation where genuinely useful

**Power BI**
- Dark, clean, executive dashboard
- Experiment overview
- Conversion and monetization
- Statistical evidence
- Revenue distribution
- Final decision

**GitHub**
- Professional README
- Notebook
- SQL
- Dashboard screenshots/link
- Case study

**Portfolio website**
- Recruiter-facing case study
- Clear business story
- GitHub and dashboard links

The final project should communicate one consistent story:

> **Data → Experiment → Evidence → Business Decision**
